# Genius API Sample Checks
Use this notebook to quickly search Genius and fetch lyrics for manual spot checks.

In [2]:
import os
import lyricsgenius

GENIUS_ACCESS_TOKEN = os.getenv("GENIUS_ACCESS_TOKEN", "")
if not GENIUS_ACCESS_TOKEN:
    raise ValueError("Set GENIUS_ACCESS_TOKEN in your environment or directly in this cell.")

genius = lyricsgenius.Genius(
    GENIUS_ACCESS_TOKEN,
    timeout=15,
    retries=3,
    remove_section_headers=True,
    skip_non_songs=True,
 )
genius.verbose = False

In [3]:
def fetch_lyrics_genius(client, title: str, artist: str) -> str:
    try:
        hit = client.search_song(title=title, artist=artist)
        if hit and hit.lyrics:
            return hit.lyrics.strip()
    except Exception as e:
        print(f"[warn] {title!r} by {artist!r}: {e}")
    return ""

In [4]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent  # go up from notebooks → project root

titles_file_path = PROJECT_ROOT / "data/processed/titles/2026/03/05/titles.csv"
lyrics_file_path = PROJECT_ROOT / "data/processed/lyrics/2026/03/05/lyrics.csv"

titles_df = pd.read_csv(titles_file_path)

In [5]:
# STEP 1a: Search Title using spotify_uri from titles.csv
spotify_uri = "5qqabIl2vWzo9ApSC317sa"

matching_row = titles_df[titles_df["spotify_uri"] == spotify_uri]
song_title = matching_row["title"].iloc[0] if not matching_row.empty else None
song_artist = matching_row["artist"].iloc[0] if not matching_row.empty else None

print(f"Spotify URI: {spotify_uri}")
print(f"Title: {song_title}")
print(f"Artist: {song_artist}")

Spotify URI: 5qqabIl2vWzo9ApSC317sa
Title: Wonderwall
Artist: Oasis


In [ ]:
# # STEP 1b: Search Lyrics using Title
# song_title = "爱怎么了"

In [6]:
# STEP 2: Fetch lyrics using Genius API
lyrics = fetch_lyrics_genius(genius, song_title, song_artist)

print("\nLyrics snippet:\n")
print((lyrics or "<no lyrics found>")[:1200])


Lyrics snippet:

Today is gonna be the day
That they're gonna throw it back to you
By now, you should've somehow
Realised what you gotta do
I don't believe that anybody
Feels the way I do about you now

Backbeat, the word is on the street
That the fire in your heart is out
I'm sure you've heard it all before
But you never really had a doubt
I don't believe that anybody
Feels the way I do about you now

And all the roads we have to walk are winding
And all the lights that lead us there are blinding
There are many things that I would like to say to you
But I don't know how

Because maybe
You're gonna be the one that saves me
And after all
You're my wonderwall

Today was gonna be the day
But they'll never throw it back to you
By now, you should've somehow
Realised what you're not to do
I don't believe that anybody
Feels the way I do about you now

And all the roads that lead you there were winding
And all the lights that light the way are blinding
There are many things that I would like 

In [7]:
# STEP 2: Get spotify_uri from titles.csv by title and artist
# If artist is empty, search only by title

song_title_tmp = '' # for manual searching if we want to try a different title

if song_title_tmp:
    song_title = song_title_tmp

if song_artist != "":
    matching_row = titles_df[
        titles_df["title"].str.contains(song_title, case=False, na=False) &
        titles_df["artist"].str.contains(song_artist, case=False, na=False)
    ]
else:
    matching_row = titles_df[
        titles_df["title"].str.contains(song_title, case=False, na=False)
    ]

result = titles_df.loc[titles_df["title"] == song_title, ["title", "spotify_uri", "artist"]]
display(result)

song_uri = result["spotify_uri"].iloc[0] if not result.empty else None
song_artist = result["artist"].iloc[0] if not result.empty else None
song_title = result["title"].iloc[0] if not result.empty else None

print(f"\nSpotify URI of the song: {song_uri}")
print(f"Artist: {song_artist}")
print(f"Title: {song_title}")

,title,spotify_uri,artist
398,Wonderwall,5qqabIl2vWzo9ApSC317sa,Oasis



Spotify URI of the song: 5qqabIl2vWzo9ApSC317sa
Artist: Oasis
Title: Wonderwall


In [8]:
# STEP 3: Update titles.csv with song_title and song_artist for the row with the matching spotify_uri (if found)
if song_uri:
    titles_df.loc[titles_df["spotify_uri"] == song_uri, ["title", "artist"]] = [song_title, song_artist]
    titles_df.to_csv(Path(titles_file_path), index=False)
    print(f"Updated titles.csv with title '{song_title}' and artist '{song_artist}' for spotify_uri '{song_uri}'.")
else:
    print(f"No matching spotify_uri found for title '{song_title}' and artist '{song_artist}'. No updates made to titles.csv.")

Updated titles.csv with title 'Wonderwall' and artist 'Oasis' for spotify_uri '5qqabIl2vWzo9ApSC317sa'.


In [9]:
# STEP 4: Update lyrics.csv with lyrics for the matching spotify_uri
lyrics_df = pd.read_csv(Path(lyrics_file_path))
if song_uri and lyrics:
    lyrics_df.loc[lyrics_df["spotify_uri"] == song_uri, "lyrics"] = lyrics
    lyrics_df.to_csv(Path(lyrics_file_path), index=False)
    print(f"Updated lyrics.csv with lyrics for spotify_uri '{song_uri}'.")
else:
    print(f"No matching spotify_uri or lyrics found for title '{song_title}' and artist '{song_artist}'. No updates made to lyrics.csv.")

Updated lyrics.csv with lyrics for spotify_uri '5qqabIl2vWzo9ApSC317sa'.
